In [0]:
# Paths to Delta tables
users_path = "/Volumes/project_2/datalake/silver/user_events/"
transactions_path = "/Volumes/project_2/datalake/silver/transactions/"
customers_path = "/Volumes/project_2/datalake/landing_zone/customers"
products_path = "/Volumes/project_2/datalake/landing_zone/products"
gold_dir = "/Volumes/project_2/datalake/gold/"

# Read Delta data
users_df = spark.read.format("delta").load(users_path)
transactions_df = spark.read.format("delta").load(transactions_path)

# Register as temp views
users_df.createOrReplaceTempView("users")
transactions_df.createOrReplaceTempView("transactions")

# Create and write gold layer tables
# dim_customer
customers_df = spark.read.option("multiLine", "true").json(customers_path)
customers_df.createOrReplaceTempView("customers_json")
dim_customer = spark.sql("""
    SELECT user_id, email, first_name, last_name, CAST(registration_date AS DATE) AS registration_date, account_type, CAST(date_of_birth AS DATE) AS date_of_birth, loyalty_points, state
    FROM customers_json
""")
dim_customer.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}dim_customer")

# dim_product
products_df = spark.read.option("multiLine", "true").json(products_path)
products_df.createOrReplaceTempView("products_json")
dim_product = spark.sql("""
    SELECT DISTINCT product_id, product_name, description, category, subcategory, brand, manufacturer, CAST(created_date AS DATE) AS created_date, is_active
    FROM products_json
""")
dim_product.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}dim_product")

# dim_date
dim_date = spark.sql("""
    SELECT DISTINCT event_date
    FROM (
        SELECT CAST(timestamp AS DATE) AS event_date FROM transactions
        UNION
        SELECT CAST(timestamp AS DATE) AS event_date FROM users
    )
""")
dim_date.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}dim_date")

# fact_transactions
fact_transactions = spark.sql("""
    SELECT transaction_id, user_id, transaction_type, CAST(timestamp AS DATE) AS event_date, status, currency, product_id, quantity, unit_price 
    FROM transactions
""")
fact_transactions.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}fact_transactions")

# fact_user_activity
fact_user_activity = spark.sql("""
    SELECT user_id, session_id, event_id, event_type, CAST(timestamp AS DATE) AS event_date, page, device, browser, country, city, search_query, element_id, product_id, quantity
    FROM users
""")
fact_user_activity.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(f"{gold_dir}fact_user_activity")

display(dim_customer)
display(dim_product)
display(dim_date)
display(fact_transactions)
display(fact_user_activity)